In [1]:
import pandas as pd
import configparser as cp
import psycopg2
from openpyxl.utils import column_index_from_string
from openpyxl.utils import get_column_letter

In [2]:
config = cp.RawConfigParser()
config_files = [
    'asuse_sheet1.properties','asuse_sheet2.properties'
    
]


print(config_files)

['asuse_sheet1.properties', 'asuse_sheet2.properties']


In [3]:
#read property file

for file in config_files:
    print('Currently Read File  :- ' + file)
    config.read(file)

    
    ip = config.get('database_details', 'database.ip')
    port = config.get('database_details', 'database.port')
    username = config.get('database_details', 'database.username')
    password = config.get('database_details', 'database.password')
    dbname = config.get('database_details', 'database.dbname')


    
    connection = psycopg2.connect(database=dbname, user=username, password=password, host=ip, port=port)
    cursor = connection.cursor()
    
    sheet_location = config.get('master_properties', 'sheet_path')
    print('reading excel file:',sheet_location)
    
    
    
    #read property asuse.tables.unique.sheets and start the nested loops
    unique_sheet_count = int(config.get('asuse_tables_for_etl', 'asuse.tables.unique.sheets'))
    
    #below loop for unique sheets only
    for i in range(unique_sheet_count):
        print('Starting for i(Unique Sheets Count):',i)
        #run below loop for blocks that are split across sheets for "each" unique sheet
        block_count = int(config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.block.count'))
        header_names = config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.header.names').split(',')
        row_seggregation_level = int(config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.rows.seggregation.level.count'))
        col_seggregation_level = int(config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.cols.seggregation.level.count'))
        row_start = int(config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.row.start'))
        row_end = int(config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.row.end'))
        col_start = column_index_from_string(config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.col.start'))
        col_end = column_index_from_string(config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.col.end'))
    
        row_seggregation_values_list = []
        row_seggregation_criteria_list = []
        col_seggregation_values_list = []
        col_seggregation_criteria_list = []
        for k in range(int(row_seggregation_level)):
            row_seggregation_values_list.append(((config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.rows.seggregation.'+str(k+1)+'.values')).split(',')))
            row_seggregation_criteria_list.append(((config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.rows.seggregation.'+str(k+1)+'.criteria'))))
        for l in range(int(col_seggregation_level)):
            col_seggregation_values_list.append(((config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.cols.seggregation.'+str(l+1)+'.values')).split(',')))
            col_seggregation_criteria_list.append(((config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.cols.seggregation.'+str(l+1)+'.criteria'))))
    
        #print('row_seggregation_criteria_list:',row_seggregation_criteria_list)
        #print('row_seggregation_values_list:',row_seggregation_values_list)
        
        #print('col_seggregation_criteria_list:',col_seggregation_criteria_list)
        #print('col_seggregation_values_list:',col_seggregation_values_list)
    
        
        for j in range(block_count):
            header_values = config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.block.'+str(j+1)+'.sheet.header.values').split(',')
            sheet_name = config.get('asuse_tables_for_etl', 'asuse.tables.table.'+str(i+1)+'.block.'+str(j+1)+'.sheet')
            input_df = pd.read_excel(sheet_location,sheet_name)
            print('Read Excel Sheet:',sheet_name)
            for i, row in enumerate(range(row_start-2, row_end-1)):
                for col in range(col_start-1,col_end):
                    insert_query_prefix = "insert into asuse_fact("+""
                    insert_query_suffix = " values("+"'"
                    for m in range(len(row_seggregation_criteria_list)):
                        if i >= len(row_seggregation_values_list[m]):
                            continue  # prevents crash

                        value = row_seggregation_values_list[m][i]

                        if value == 'NULL':
                            continue  

                        insert_query_prefix += row_seggregation_criteria_list[m] + ","
                        insert_query_suffix += value + "','"
                    for n in range(len(col_seggregation_criteria_list)):
                        if col_seggregation_values_list[n][col-col_start+1] == 'NULL' :
                             continue
                        insert_query_prefix = insert_query_prefix + col_seggregation_criteria_list[n] +","
                        insert_query_suffix = insert_query_suffix + col_seggregation_values_list[n][col-col_start+1]+"','"
                        #print('col_seggregation_criteria_list['+str(n)+']:',col_seggregation_criteria_list[n],' col_seggregation_values_list['+str(n)+']['+str(col-col_start+1)+']:',col_seggregation_values_list[n][col-col_start])
                    
                    
                    indicator_val = input_df.iloc[row].values[col]
                   
                    # print('Inserting, indicator_val['+str(row+2)+']['+str(get_column_letter(col+1))+']:',indicator_val)
    
                    for o in range(len(header_names)):
                        insert_query_prefix = insert_query_prefix + header_names[o] +","
                        insert_query_suffix = insert_query_suffix + header_values[o]+"','"
    
                    insert_query_prefix = insert_query_prefix  +"indicator_value,"
                    # rounded_number = round(float(indicator_val), 2)
                    insert_query_suffix = insert_query_suffix + str(indicator_val)+"','"
    
                    insert_query_prefix = insert_query_prefix  +"created_by,"
                    insert_query_suffix = insert_query_suffix + 'system' +"','"
                    insert_query_prefix = insert_query_prefix  +"updated_by,"
                    insert_query_suffix = insert_query_suffix + ' system' +"','"
    
                    
                    # insert_query_prefix = insert_query_prefix  +"year,"
                    # insert_query_suffix = insert_query_suffix + sheet_year +"','"
    
    
                    asuse_fact_code = 'asuse_'+'_'+sheet_name+'_'+str(row+2)+'_'+str(get_column_letter(col+1))+'_'+ str(indicator_val)
                    asuse_fact_code = asuse_fact_code.replace(' ', '')
                    insert_query_prefix = insert_query_prefix + "asuse_fact_code)"
                    insert_query_suffix = insert_query_suffix + asuse_fact_code+"')"
    
                    final_query = insert_query_prefix+insert_query_suffix
                    print(final_query)
                    cursor.execute(final_query)
                     
                    connection.commit()

connection.close()
print('Data insert successfully')



Currently Read File  :- asuse_sheet1.properties
reading excel file: data\Data for eSankhyiki-QBUSE.xlsx
Starting for i(Unique Sheets Count): 0
Read Excel Sheet: Table-1
insert into asuse_fact(indicator_code,sector_code,financial_year,quarter_code,frequency_code,state_code,indicator_value,created_by,updated_by,asuse_fact_code) values('40','1','2026','1','2','37','99.03','system',' system','asuse__Table-1_4_N_99.03')
insert into asuse_fact(indicator_code,sector_code,financial_year,quarter_code,frequency_code,state_code,indicator_value,created_by,updated_by,asuse_fact_code) values('40','2','2026','1','2','37','98.85','system',' system','asuse__Table-1_4_O_98.85')
insert into asuse_fact(indicator_code,sector_code,financial_year,quarter_code,frequency_code,state_code,indicator_value,created_by,updated_by,asuse_fact_code) values('40','3','2026','1','2','37','98.95','system',' system','asuse__Table-1_4_P_98.95')
insert into asuse_fact(indicator_code,sector_code,financial_year,quarter_code,fre